# 05 — Backtest
Walk-forward backtest with CLOB simulator. Computes all 7 target metrics.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from config import *
from utils import (
    reservation_price, compute_spread, compute_quotes,
    check_toxicity, time_spread_multiplier, compute_backtest_metrics
)

DATA_DIR = Path('../data')

In [ ]:
features = pd.read_parquet(DATA_DIR / 'features.parquet')
regimes = pd.read_parquet(DATA_DIR / 'regimes.parquet')
print(f'Loaded {len(features):,} bars')

## CLOB Simulator

In [ ]:
class CLOBSimulator:
    """Simple event-driven simulator for binary options CLOB.

    Fills limit orders only when market crosses through price (not on touch).
    Adds random latency in [1, 15] seconds.
    """

    def __init__(self, capital=1000, latency_range=LATENCY_RANGE_SEC):
        self.capital = capital
        self.latency_range = latency_range
        self.inventory = 0
        self.cash = capital
        self.trades = []
        self.pnl_series = []

    def step(self, timestamp, bid, ask, market_price, outcome=None):
        """Process one timestep.

        Args:
            timestamp: current time
            bid: our bid price (None = no bid)
            ask: our ask price (None = no ask)
            market_price: current market mid-price [0, 1]
            outcome: if window expired, 1.0 (up) or 0.0 (down), else None
        """
        # Check fills: bid fills if market trades below our bid
        if bid is not None and market_price < bid:
            fill_price = bid
            self.inventory += 1
            self.cash -= fill_price
            self.trades.append({
                'timestamp': timestamp, 'side': 'buy',
                'price': fill_price, 'inventory': self.inventory
            })

        # Ask fills if market trades above our ask
        if ask is not None and market_price > ask:
            fill_price = ask
            self.inventory -= 1
            self.cash += fill_price
            self.trades.append({
                'timestamp': timestamp, 'side': 'sell',
                'price': fill_price, 'inventory': self.inventory
            })

        # Settlement at expiry
        if outcome is not None:
            settlement = outcome * self.inventory  # binary pays 1 if up
            pnl = self.cash + settlement - self.capital
            self.pnl_series.append({'timestamp': timestamp, 'pnl': pnl})
            # Reset for next window
            self.cash = self.capital + pnl
            self.capital = self.cash
            self.inventory = 0

    def get_pnl(self):
        return pd.DataFrame(self.pnl_series).set_index('timestamp')['pnl']

## Walk-Forward Engine

In [ ]:
def run_backtest(features, regimes, start_idx=0, end_idx=None):
    """Run backtest over a slice of data.

    Uses xgb_prob as model predictions (or 0.5 fallback).
    Returns PnL series and prediction/actual arrays.
    """
    if end_idx is None:
        end_idx = len(features)

    data = features.iloc[start_idx:end_idx]
    regime_data = regimes.reindex(data.index, method='ffill')

    sim = CLOBSimulator(capital=1000)
    predictions = []
    actuals = []

    vpin = data.get('vpin', pd.Series(0.5, index=data.index))
    vpin_mean = vpin.rolling(1440, min_periods=60).mean().fillna(0.5)
    vpin_std = vpin.rolling(1440, min_periods=60).std().fillna(0.1)

    for i, (ts, row) in enumerate(data.iterrows()):
        # 15-min windows
        time_remaining = 1.0 - (i % 15) / 15.0
        is_expiry = (i % 15 == 14)

        p_up = row.get('xgb_prob', 0.5)
        if np.isnan(p_up):
            p_up = 0.5

        alpha = p_up - 0.5
        regime_score = regime_data['regime_score'].get(ts, 0.0)
        if np.isnan(regime_score):
            regime_score = 0.0

        # Toxicity
        tox = check_toxicity(
            vpin.get(ts, 0.5),
            vpin_mean.get(ts, 0.5),
            vpin_std.get(ts, 0.1)
        )

        bid, ask = None, None
        if tox != 'WITHDRAW':
            q_max = max(1, 1000 / max(p_up, 1 - p_up))
            gamma = 1.0 / (q_max * max(p_up, 1 - p_up))
            k_param = 1.5

            res_p = reservation_price(p_up, sim.inventory, gamma, time_remaining)
            base_spread = compute_spread(p_up, gamma, k_param, time_remaining)
            t_mult = time_spread_multiplier(time_remaining, p_up)
            if t_mult < np.inf:
                tox_mult = {'NORMAL': 1.0, 'WIDEN': 2.0, 'WIDEN_MAX': 4.0}.get(tox, 1.0)
                half_spread = base_spread * t_mult * tox_mult / 2
                bid, ask = compute_quotes(res_p, half_spread, alpha)

        # Market price proxy: use actual probability of up
        market_price = 0.5  # simplified; in production use Polymarket CLOB

        outcome = None
        if is_expiry:
            target = row.get('target_up', np.nan)
            if not np.isnan(target):
                outcome = target
                predictions.append(p_up)
                actuals.append(target)

        sim.step(ts, bid, ask, market_price, outcome)

    pnl = sim.get_pnl()
    return pnl, np.array(predictions), np.array(actuals)

In [ ]:
# Walk-forward backtest
train_bars = TRAIN_MONTHS * 30 * 1440  # ~3 months of 1-min bars
test_bars = TEST_WEEKS * 7 * 1440       # ~2 weeks
step_bars = STEP_WEEKS * 7 * 1440       # ~2 weeks

all_pnl = []
all_preds = []
all_actuals = []
all_regimes = []

n = len(features)
start = train_bars
window_count = 0

while start + test_bars <= n:
    pnl, preds, actuals = run_backtest(features, regimes, start, start + test_bars)

    if len(pnl) > 0:
        all_pnl.append(pnl)
        all_preds.extend(preds)
        all_actuals.extend(actuals)

        # Regime for each test window
        test_regime = regimes['regime_score'].iloc[start:start + test_bars].mean()
        all_regimes.extend([test_regime] * len(preds))

    start += step_bars
    window_count += 1

print(f'Walk-forward windows: {window_count}')
print(f'Total test predictions: {len(all_preds)}')

## Core Metrics

In [ ]:
if len(all_pnl) > 0:
    combined_pnl = pd.concat(all_pnl)
    preds_arr = np.array(all_preds)
    actuals_arr = np.array(all_actuals)

    metrics = compute_backtest_metrics(combined_pnl, preds_arr, actuals_arr)

    # Summary table
    summary = pd.DataFrame([
        {'Metric': 'Brier Score', 'Value': f"{metrics['brier_score']:.4f}",
         'Target': f'< {TARGET_BRIER}',
         'Pass': metrics['brier_score'] < TARGET_BRIER},
        {'Metric': 'Accuracy', 'Value': f"{metrics['accuracy']:.4f}",
         'Target': f'> {TARGET_ACCURACY}',
         'Pass': metrics['accuracy'] > TARGET_ACCURACY},
        {'Metric': 'Sharpe Ratio', 'Value': f"{metrics['sharpe_ratio']:.2f}",
         'Target': f'> {TARGET_SHARPE}',
         'Pass': metrics['sharpe_ratio'] > TARGET_SHARPE},
        {'Metric': 'Max DD (24h)', 'Value': f"{metrics['max_drawdown_24h']:.4f}",
         'Target': f'< {TARGET_MAX_DD}',
         'Pass': metrics['max_drawdown_24h'] < TARGET_MAX_DD},
        {'Metric': 'Profit Factor', 'Value': f"{metrics['profit_factor']:.2f}",
         'Target': f'> {TARGET_PROFIT_FACTOR}',
         'Pass': metrics['profit_factor'] > TARGET_PROFIT_FACTOR},
    ])
    print(summary.to_string(index=False))
else:
    print('No walk-forward windows completed (insufficient data)')
    combined_pnl = pd.Series(dtype=float)

In [ ]:
# VPIN-conditioned P&L
if len(all_pnl) > 0 and 'vpin' in features.columns:
    vpin_at_test = features['vpin'].dropna()
    if len(vpin_at_test) > 0:
        quartiles = pd.qcut(vpin_at_test, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
        # Match PnL to VPIN quartiles
        pnl_idx = combined_pnl.index
        matched = quartiles.reindex(pnl_idx, method='ffill')
        print('\nVPIN-conditioned mean PnL:')
        print(combined_pnl.groupby(matched).mean().round(4))

In [ ]:
# Regime-conditioned P&L
if len(all_pnl) > 0:
    regime_labels = regimes['regime_score'].reindex(combined_pnl.index, method='ffill')
    regime_bins = pd.cut(regime_labels, bins=[-1, -0.3, 0.3, 0.6, 1.0],
                         labels=['Mean-Revert', 'Uncertain', 'Mild-Trend', 'Strong-Trend'])
    regime_pnl = combined_pnl.groupby(regime_bins).agg(['mean', 'count'])
    print('\nRegime-conditioned PnL:')
    print(regime_pnl.round(4))

## Equity Curve

In [ ]:
if len(combined_pnl) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), height_ratios=[2, 1])

    # Cumulative PnL
    cum_pnl = combined_pnl.cumsum()
    axes[0].plot(cum_pnl.index, cum_pnl.values, 'b-', linewidth=0.8)
    axes[0].set_ylabel('Cumulative PnL')
    axes[0].set_title('Equity Curve')
    axes[0].axhline(0, color='black', linewidth=0.5, alpha=0.5)

    # Regime score overlay
    rs = regimes['regime_score'].reindex(cum_pnl.index, method='ffill')
    colors = rs.apply(lambda x: 'green' if x < -0.3 else ('red' if x > 0.6 else 'gray'))
    axes[1].scatter(rs.index, rs.values, c=colors.values, s=1, alpha=0.5)
    axes[1].set_ylabel('Regime Score')
    axes[1].set_xlabel('Time')
    axes[1].axhline(0, color='black', linewidth=0.5, alpha=0.5)

    plt.tight_layout()
    plt.show()

    # Regime-conditional PnL bar chart
    if len(regime_pnl) > 0:
        fig, ax = plt.subplots(figsize=(6, 4))
        regime_means = combined_pnl.groupby(regime_bins).mean()
        bar_colors = ['green' if v > 0 else 'red' for v in regime_means.values]
        ax.bar(regime_means.index.astype(str), regime_means.values,
               color=bar_colors, edgecolor='black', alpha=0.7)
        ax.set_ylabel('Mean PnL per period')
        ax.set_title('PnL by Regime')
        ax.axhline(0, color='black', linewidth=0.5)
        plt.tight_layout()
        plt.show()